In [1]:
import pandas as pd
import numpy as np
import string
import warnings
import logging
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import nltk
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings('ignore')
logging.getLogger('lightgbm').setLevel(logging.ERROR)
logging.getLogger('catboost').setLevel(logging.ERROR)

# Download necessary NLTK data files
nltk.download('wordnet')

# Preprocessing function without removing stopwords
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Lemmatize words
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

# Load and prepare the training data
train_df = pd.read_csv("/Users/eren/Desktop/412Project/bugs-train.csv")
test_df = pd.read_csv("/Users/eren/Desktop/412Project/bugs-test.csv")
severity_mapping = {
    'enhancement': 1,
    'trivial': 2,
    'minor': 3,
    'normal': 4,
    'major': 5,
    'blocker': 6,
    'critical': 7
}
train_df['severity'] = train_df['severity'].map(severity_mapping)

# Apply preprocessing to the text data
train_df['summary'] = train_df['summary'].apply(preprocess_text)
test_df['summary'] = test_df['summary'].apply(preprocess_text)

# Prepare text data
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(train_df['summary'])
X_test_tfidf = vectorizer.transform(test_df['summary'])

# Define the base models
base_models = {
    'xgboost': XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    'lightgbm': LGBMClassifier(),
    'catboost': CatBoostClassifier(verbose=0)
}

param_grids = {
    'xgboost': {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 5, 7]
    },
    'lightgbm': {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 5, 7]
    },
    'catboost': {
        'iterations': [100, 200],
        'learning_rate': [0.01, 0.1],
        'depth': [3, 5, 7]
    }
}

# Train separate models for each class using stratified sampling
models = {model_name: {} for model_name in base_models.keys()}
for severity, label in tqdm(severity_mapping.items(), desc="Training models"):
    y_binary = train_df['severity'] == label
    X_train, X_val, y_train, y_val = train_test_split(X_train_tfidf, y_binary, test_size=0.1, random_state=42, stratify=y_binary)
    
    for model_name, model in base_models.items():
        grid_search = GridSearchCV(model, param_grids[model_name], scoring='f1', cv=3, verbose=0)
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        models[model_name][label] = best_model

        # Evaluate model on the validation set (optional, can be removed)
        y_pred = best_model.predict(X_val)
        print(f"{model_name} Class {label} - Accuracy: {accuracy_score(y_val, y_pred)}, "
              f"Precision: {precision_score(y_val, y_pred, average='macro')}, "
              f"Recall: {recall_score(y_val, y_pred, average='macro')}, "
              f"F1 Score: {f1_score(y_val, y_pred, average='macro')}")

# Meta-learner: Train an SVM model using the predictions of all base models
def train_meta_learner(models, X, y):
    meta_features = np.hstack([
        np.column_stack([model[label].predict_proba(X)[:, 1] for model in models.values()])
        for label in severity_mapping.values()
    ])
    meta_learner = SVC(kernel='linear', probability=True)
    
    # Normalize class labels for the meta-learner
    y_normalized = y - 1
    
    meta_learner.fit(meta_features, y_normalized)
    return meta_learner

# Prepare meta-features for the meta-learner
meta_features_train = np.hstack([
    np.column_stack([model[label].predict_proba(X_train_tfidf)[:, 1] for model in models.values()])
    for label in severity_mapping.values()
])
meta_features_test = np.hstack([
    np.column_stack([model[label].predict_proba(X_test_tfidf)[:, 1] for model in models.values()])
    for label in severity_mapping.values()
])

# Train the meta-learner
meta_learner = train_meta_learner(models, X_train_tfidf, train_df['severity'])

# Predict with meta-learner
predictions_normalized = meta_learner.predict(meta_features_test)
final_predictions = predictions_normalized + 1

# Evaluate the ensemble model
ensemble_accuracy = accuracy_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1)
ensemble_precision = precision_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1, average='macro')
ensemble_recall = recall_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1, average='macro')
ensemble_f1 = f1_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1, average='macro')

print(f"Ensemble Model - Accuracy: {ensemble_accuracy}, "
      f"Precision: {ensemble_precision}, "
      f"Recall: {ensemble_recall}, "
      f"F1 Score: {ensemble_f1}")

# Map numeric labels back to string labels
reverse_severity_mapping = {v: k for k, v in severity_mapping.items()}
predicted_severities = [reverse_severity_mapping[label] for label in final_predictions]

# Create a submission DataFrame
submission_df = pd.DataFrame({
    'bug_id': test_df['bug_id'],
    'severity': predicted_severities  # Change column name to 'severity'
})

# Save to CSV
submission_path = "/Users/eren/Desktop/412Project/submission5555.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission file saved to {submission_path}")

[nltk_data] Downloading package wordnet to /Users/eren/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Training models:   0%|                                    | 0/7 [00:00<?, ?it/s]

xgboost Class 1 - Accuracy: 0.9725625, Precision: 0.8078541938481887, Recall: 0.5099973141803875, F1 Score: 0.5127348995511737
[LightGBM] [Info] Number of positive: 2655, number of negative: 93343
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.240383 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 141211
[LightGBM] [Info] Number of data points in the train set: 95998, number of used features: 3232
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.027657 -> initscore=-3.559836
[LightGBM] [Info] Start training from score -3.559836
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [

Training models:  14%|███▌                     | 1/7 [12:08<1:12:50, 728.36s/it]

catboost Class 1 - Accuracy: 0.9723125, Precision: 0.7363630676842237, Recall: 0.5076756980918202, F1 Score: 0.5082951264263552
xgboost Class 2 - Accuracy: 0.992625, Precision: 0.8297069734483764, Recall: 0.5166036943744753, F1 Score: 0.5298949932820799
[LightGBM] [Info] Number of positive: 722, number of negative: 95276
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.282510 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 141162
[LightGBM] [Info] Number of data points in the train set: 95998, number of used features: 3233
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.007521 -> initscore=-4.882508
[LightGBM] [Info] Start training from score -4.882508
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

Training models:  29%|███████▋                   | 2/7 [22:23<55:09, 661.84s/it]

...

Training models:  86%|███████████████████████▏   | 6/7 [59:01<09:28, 568.88s/it]

catboost Class 6 - Accuracy: 0.9955, Precision: 0.497812226528316, Recall: 0.4999372253609542, F1 Score: 0.4988724630418441
xgboost Class 7 - Accuracy: 0.951125, Precision: 0.8853583877036035, Recall: 0.8741886530560121, F1 Score: 0.8796588667427776
[LightGBM] [Info] Number of positive: 11194, number of negative: 84804
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.331937 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 141005
[LightGBM] [Info] Number of data points in the train set: 95998, number of used features: 3225
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116607 -> initscore=-2.024965
[LightGBM] [Info] Start training from score -2.024965
[LightGBM] [Info] Number of positive: 11195, number of negative: 84804
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.285397 seconds.
You can set `

Training models: 100%|█████████████████████████| 7/7 [1:09:21<00:00, 594.53s/it]

catboost Class 7 - Accuracy: 0.95, Precision: 0.8836045223779759, Recall: 0.8695980790810844, F1 Score: 0.8764195865532307
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).


[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^m